# ADMM TEST

In [ ]:
import time
import struct
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================
# IMPORTANT: Update this path to your actual bitstream location
bitstream_path = '/home/xilinx/admm/admm.bit' 
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

CONTROL_ADDR   = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE     = 0x10000

print(f"Manually mapping Control   : {hex(CONTROL_ADDR)}")
print(f"Manually mapping Control_R : {hex(CONTROL_R_ADDR)}")

control_ip   = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

In [ ]:
import time
import struct
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================
# IMPORTANT: Update this path to your actual bitstream location
bitstream_path = '/home/xilinx/admm/admm.bit' 
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

CONTROL_ADDR   = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE     = 0x10000

print(f"Manually mapping Control   : {hex(CONTROL_ADDR)}")
print(f"Manually mapping Control_R : {hex(CONTROL_R_ADDR)}")

control_ip   = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

# =====================================================================
# 2. Problem Setup (15x15 Case - Reverse Engineered)
# =====================================================================
print("\nRunning ADMM accelerator testbench (15x15)...")

num_rows = 15
num_cols = 15
PACK_SIZE = 16
MAX_NNZ_WORDS = 16 # Padding max words for AXI bursts

# OSQP Constants
OSQP_RHO = 0.1
OSQP_RHO_EQ_OVER_RHO_INEQ = 1000.0
OSQP_RHO_TOL = 1e-4
OSQP_INFTY = 1e20
OSQP_MIN_SCALING = 1e-4
OSQP_RHO_MAX = 1e6
OSQP_RHO_MIN = 1e-6

np.random.seed(123) # Match C++ rng seed

# 1. Generate Ground Truth x* and Matrix P
x_true = np.random.uniform(-2.0, 2.0, num_cols).astype(np.float32)
P_values = np.random.uniform(1.0, 3.0, num_cols).astype(np.float32)
P_row_idx = np.arange(num_cols, dtype=np.int32)
P_col_ptr = np.arange(num_cols + 1, dtype=np.int32)
P_diag = P_values.copy()
P_nnz = num_cols
q = (-P_values * x_true).astype(np.float32)

# 2. Generate Random Matrix A and compute z* = A * x*
A_values = np.random.uniform(-1.0, 1.0, num_cols).astype(np.float32)
A_values[np.abs(A_values) < 1e-3] = 1e-3
A_row_idx = np.random.randint(0, num_rows, num_cols).astype(np.int32)
A_col_ptr = np.arange(num_cols + 1, dtype=np.int32)
A_nnz = num_cols

z_true = np.zeros(num_rows, dtype=np.float32)
for c in range(num_cols):
    z_true[A_row_idx[c]] += A_values[c] * x_true[c]

# Transpose A
A_sparse = sp.csc_matrix((A_values, A_row_idx, A_col_ptr), shape=(num_rows, num_cols))
AT_sparse = A_sparse.transpose().tocsc()
AT_col_ptr = AT_sparse.indptr.astype(np.int32)
AT_row_idx = AT_sparse.indices.astype(np.int32)
AT_values  = AT_sparse.data.astype(np.float32)

# 3. Assign Bounds based on z*
l_in = np.zeros(num_rows, dtype=np.float32)
u_in = np.zeros(num_rows, dtype=np.float32)
rho_in = np.zeros(num_rows, dtype=np.float32)

constraint_types = np.random.randint(0, 3, num_rows)
slack1 = np.random.uniform(0.5, 5.0, num_rows).astype(np.float32)
slack2 = np.random.uniform(0.5, 5.0, num_rows).astype(np.float32)

for i in range(num_rows):
    ctype = constraint_types[i]
    if ctype == 0:
        l_in[i] = z_true[i]
        u_in[i] = z_true[i]
        rho_in[i] = OSQP_RHO * OSQP_RHO_EQ_OVER_RHO_INEQ
    elif ctype == 1:
        l_in[i] = z_true[i] - slack1[i]
        u_in[i] = z_true[i] + slack2[i]
        rho_in[i] = OSQP_RHO
    else:
        l_in[i] = -OSQP_INFTY
        u_in[i] = OSQP_INFTY
        rho_in[i] = OSQP_RHO_MIN
        
    rho_in[i] = np.clip(rho_in[i], OSQP_RHO_MIN, OSQP_RHO_MAX)

# Execution parameters
alpha = 1.6
sigma = 1e-6
admm_max_iterations = 2000
pcg_max_iterations = 500

print(f"Problem Size: {num_rows}x{num_cols}")
print("Data matrices generated successfully.")

# 
# Export Matrix and Vector data to use same datasets in C test
#

import os
import numpy as np

# 1. Define the target directory
output_dir = "/home/xilinx/admm/baremetal/data"

# 2. Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Ensure your arrays are the exact sizes and types C expects
x_true = x_true.astype(np.float32)
P_values = P_values.astype(np.float32)
P_col_ptr = P_col_ptr.astype(np.int32)
A_values = A_values.astype(np.float32)
A_col_ptr = A_col_ptr.astype(np.int32)
A_row_idx = A_row_idx.astype(np.int32)
l_in = l_in.astype(np.float32)
u_in = u_in.astype(np.float32)
q = q.astype(np.float32)
rho_in = rho_in.astype(np.float32)

# 3. Dump to raw binary files in the specific directory
x_true.tofile(os.path.join(output_dir, "x_true.bin"))
P_values.tofile(os.path.join(output_dir, "P_values.bin"))
P_col_ptr.tofile(os.path.join(output_dir, "P_col_ptr.bin"))
A_values.tofile(os.path.join(output_dir, "A_values.bin"))
A_col_ptr.tofile(os.path.join(output_dir, "A_col_ptr.bin"))
A_row_idx.tofile(os.path.join(output_dir, "A_row_idx.bin"))
l_in.tofile(os.path.join(output_dir, "l_in.bin"))
u_in.tofile(os.path.join(output_dir, "u_in.bin"))
q.tofile(os.path.join(output_dir, "q.bin"))
rho_in.tofile(os.path.join(output_dir, "rho_in.bin"))

print(f"Matrices successfully dumped to: {output_dir}")


# =====================================================================
# 3. Buffer Allocation and Packing
# =====================================================================
def pack_csc_to_words(row_idx, values):
    nnz = len(row_idx)
    # Cacheable=False bypasses Zynq ARM cache sync issues
    row_words = allocate(shape=(MAX_NNZ_WORDS, PACK_SIZE), dtype=np.int32, cacheable=False)
    val_words = allocate(shape=(MAX_NNZ_WORDS, PACK_SIZE), dtype=np.float32, cacheable=False)
    
    row_words[:] = 0
    val_words[:] = 0.0
    
    row_words.reshape(-1)[:nnz] = row_idx
    val_words.reshape(-1)[:nnz] = values
    return row_words, val_words

print("Allocating and Packing Hardware CMA Buffers (Uncached)...")

A_row_words, A_val_words   = pack_csc_to_words(A_row_idx, A_values)
AT_row_words, AT_val_words = pack_csc_to_words(AT_row_idx, AT_values)
P_row_words, P_val_words   = pack_csc_to_words(P_row_idx, P_values)

# Pad max words to ensure AXI bursts don't read out of bounds
PAD = 16 

A_col_ptr_hw  = allocate(shape=(num_cols + 1 + PAD,), dtype=np.int32, cacheable=False); A_col_ptr_hw[:num_cols+1]  = A_col_ptr
AT_col_ptr_hw = allocate(shape=(num_rows + 1 + PAD,), dtype=np.int32, cacheable=False); AT_col_ptr_hw[:num_rows+1] = AT_col_ptr
P_col_ptr_hw  = allocate(shape=(num_cols + 1 + PAD,), dtype=np.int32, cacheable=False); P_col_ptr_hw[:num_cols+1]  = P_col_ptr

P_diag_hw = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False); P_diag_hw[:num_cols] = P_diag
l_in_hw   = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False); l_in_hw[:num_rows]   = l_in
u_in_hw   = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False); u_in_hw[:num_rows]   = u_in
q_in_hw   = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False); q_in_hw[:num_cols]   = q
rho_in_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False); rho_in_hw[:num_rows] = rho_in

x_out_hw  = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False); x_out_hw[:] = 0.0
y_out_hw  = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False); y_out_hw[:] = 0.0

# =====================================================================
# 4. Hardware Execution
# =====================================================================
def write_64bit_address(ip, base_offset, address):
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

def float_to_uint(f_val):
    return struct.unpack('<I', struct.pack('<f', f_val))[0]

def uint_to_float(u_val):
    return struct.unpack('<f', struct.pack('<I', u_val))[0]

print("Configuring Registers...")

# Write to Control Registers (0xA0000000)
control_ip.write(0x10, num_rows)
control_ip.write(0x18, num_cols)
control_ip.write(0x20, A_nnz)
control_ip.write(0x28, P_nnz)
control_ip.write(0x30, float_to_uint(sigma))
control_ip.write(0x38, float_to_uint(alpha))
control_ip.write(0x40, admm_max_iterations)
control_ip.write(0x48, pcg_max_iterations)

# Write to Control_R Registers (Addresses to 0xA0010000)
write_64bit_address(control_r_ip, 0x10, A_row_words.device_address)
write_64bit_address(control_r_ip, 0x1c, A_col_ptr_hw.device_address)
write_64bit_address(control_r_ip, 0x28, A_val_words.device_address)

write_64bit_address(control_r_ip, 0x34, AT_row_words.device_address)
write_64bit_address(control_r_ip, 0x40, AT_col_ptr_hw.device_address)
write_64bit_address(control_r_ip, 0x4c, AT_val_words.device_address)

write_64bit_address(control_r_ip, 0x58, P_row_words.device_address)
write_64bit_address(control_r_ip, 0x64, P_col_ptr_hw.device_address)
write_64bit_address(control_r_ip, 0x70, P_val_words.device_address)

write_64bit_address(control_r_ip, 0x7c, P_diag_hw.device_address)
write_64bit_address(control_r_ip, 0x88, l_in_hw.device_address)
write_64bit_address(control_r_ip, 0x94, u_in_hw.device_address)
write_64bit_address(control_r_ip, 0xa0, q_in_hw.device_address)
write_64bit_address(control_r_ip, 0xac, rho_in_hw.device_address)

write_64bit_address(control_r_ip, 0xb8, x_out_hw.device_address)
write_64bit_address(control_r_ip, 0xc4, y_out_hw.device_address)

print("Starting Hardware Accelerator...")
hw_start = time.time()

# Send Start Signal (Bit 0 of 0x00)
control_ip.write(0x00, 0x01) 

# Wait for ap_done (Bit 1 of 0x00)
while (control_ip.read(0x00) & 0x02) == 0:
    pass

hw_end = time.time()
print(f"HW Execution Time: {(hw_end - hw_start) * 1000:.4f} ms")

# Read Scalar Outputs
admm_iters = control_ip.read(0x50)
pcg_iters  = control_ip.read(0x60)
r_prim_out = uint_to_float(control_ip.read(0x70))
r_dual_out = uint_to_float(control_ip.read(0x80))
status_out = control_ip.read(0x90)

# =====================================================================
# 5. Software Reference & Verification
# =====================================================================
print("\n--- Simulation Results ---")
print(f"Status: {'Converged' if status_out == 1 else 'Max Iterations Reached'}")
print(f"ADMM Iterations: {admm_iters}")
print(f"Total PCG Iterations: {pcg_iters}")
print(f"Primal Residual: {r_prim_out:.5e}")
print(f"Dual Residual: {r_dual_out:.5e}")

x_hw = np.array(x_out_hw[:num_cols])
mae = np.mean(np.abs(x_hw - x_true))

print(f"\nMean Absolute Error from x_true: {mae:.5e}")
print("--- Full x_out vs Expected ---")
for i in range(num_cols):
    print(f"x[{i:2}]: {x_hw[i]:13.5f} | Expected: {x_true[i]:13.5f}")

if status_out == 1 and mae < 1e-2:
    print("\n>>> SUCCESS: Problem converged perfectly to the ground truth! <<<")
else:
    print("\n>>> FAILED: Did not converge to the expected target. <<<")

# Clean up memory
for buf in [A_row_words, A_val_words, AT_row_words, AT_val_words, P_row_words, P_val_words,
            A_col_ptr_hw, AT_col_ptr_hw, P_col_ptr_hw, P_diag_hw, l_in_hw, u_in_hw, 
            q_in_hw, rho_in_hw, x_out_hw, y_out_hw]:
    buf.freebuffer()